# 調査の実行（本番ジョブ用）

アプリが Volumes に置いた調査定義を受け取り、

```
validate → panel → screen → run → aggregate（export 含む）→ metadata
```

を1本で通します。**このノートブックは単一の `notebook_task` としてジョブに登録し、**
**`job_parameters` の `survey` に調査定義（Volumes 上の YAML）のパスを渡します**
（`deploy/README.md` §3 / `docs/SPEC_UI.md` §6.1）。

`quickstart.ipynb` と違い、`personas_base` の構築やデモ用の `fake` エンドポイントへの
切り替えは含みません。本番実行が前提で、`model.endpoint` は調査定義側が決めます。

**どこかで失敗したら例外を送出してジョブを失敗させます。** 握りつぶして先に進めると、
実行できていないのに `runs` に記録が残ったかのように見えてしまうため。

## アプリを経由しない単体実行

`survey`（と、必要なら `catalog` / `schema` / `warehouse`）のウィジェットさえ埋まれば、
このノートブックはアプリの「調査を開始する」ボタンを経由しなくても動きます。
UI 側（`persona_sim/uiconfig/jobs.py` の投入経路。`tests/test_jobs.py` で確認済み）と
ノートブック側を切り分けてテストしたいときは、次のどちらかで直接起動してください。

1. クラスタにアタッチしてこのノートブックを開き、上部のウィジェットに値を入れて全セル実行する
2. 事前定義済みのジョブに対して、Databricks Jobs UI の「今すぐ実行」（または
   `databricks jobs run-now --job-id <ID> --job-parameters survey=...`）で
   `job_parameters` を直接入力して起動する

どちらの経路でも Volumes 上に調査定義 YAML が必要です。手早く無料で試すには
`examples/survey_sample_smoke.yaml`（`model.endpoint: fake`・小規模）を Volumes に置いて
使ってください。手順は `deploy/README.md` の「ノートブック単体の動作確認」を参照。

In [0]:
# persona_sim の場所を通す。
import sys
from importlib.util import find_spec
from pathlib import Path

if find_spec("persona_sim") is None:
    repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    if (repo_root / "persona_sim").is_dir():
        sys.path.insert(0, str(repo_root))

if find_spec("persona_sim") is None:
    raise ImportError(
        "persona_sim を import できない。このノートブックはリポジトリごと "
        "Databricks Git フォルダに置いて実行すること（deploy/README.md §3）"
    )

import persona_sim

print("persona_sim", persona_sim.__version__)


In [0]:
import os

dbutils.widgets.text("survey", "", "調査定義のパス（Volumes 上の YAML）")  # noqa: F821
dbutils.widgets.text("catalog", "", "任意: PERSONA_SIM_CATALOG（クラスタ未設定時の上書き）")  # noqa: F821
dbutils.widgets.text("schema", "", "任意: PERSONA_SIM_SCHEMA（クラスタ未設定時の上書き）")  # noqa: F821
dbutils.widgets.text("warehouse", "", "任意: PERSONA_SIM_WAREHOUSE（クラスタ未設定時の上書き）")  # noqa:F821

In [0]:
survey_path = dbutils.widgets.get("survey")  # noqa: F821
if not survey_path:
    raise ValueError(
        "ジョブパラメータ survey が空。Volumes 上の調査定義 YAML のパスを渡すこと"
        "（アプリ経由でも、ジョブの『今すぐ実行』で直接指定してもよい）"
    )

# クラスタ側に PERSONA_SIM_CATALOG 等が未設定でも単体実行できるよう、
# ウィジェットに値があればその場で環境変数を上書きする（空なら何もしない）。
for widget, env_var in (
    ("catalog", "PERSONA_SIM_CATALOG"),
    ("schema", "PERSONA_SIM_SCHEMA"),
    ("warehouse", "PERSONA_SIM_WAREHOUSE"),
):
    value = dbutils.widgets.get(widget)  # noqa: F821
    if value:
        os.environ[env_var] = value

print("調査定義:", survey_path)

## 1. 調査定義を読み、検証する

エラーがあれば実行前にここで止める（`SPEC.md` §5）。

In [0]:
from persona_sim.config import output_dir, storage_config
from persona_sim.panel.loader import load_survey
from persona_sim.panel.validate import validate_feasibility, validate_static
from persona_sim.spark import get_spark
from persona_sim.storage import delta
from persona_sim.storage.locator import PERSONAS_BASE, locator

survey = load_survey(survey_path)
storage = storage_config()
spark = get_spark()

report = validate_static(survey)

# 抽出可能性（E1・§4.1）。割り付けが埋まらない調査を、LLM を呼ぶ前に止める。
personas_locator = locator(PERSONAS_BASE, storage)
if delta.table_exists(spark, personas_locator):
    feasibility = validate_feasibility(spark, delta.read_table(spark, personas_locator), survey)
    report.errors.extend(feasibility.errors)
    report.warnings.extend(feasibility.warnings)
else:
    print("警告", f"{personas_locator.describe()} が無いので抽出可能性を確認できない")

for issue in report.warnings:
    print("警告", issue)
if not report.ok:
    for issue in report.errors:
        print("エラー", issue)
    raise ValueError(f"調査定義にエラーがある: {survey.survey_id}")

print("調査:", survey.survey_id, "(", survey.name, ")")

## 2. `panel` — 割り付けどおりに人を選ぶ

In [0]:
from persona_sim.panel.build import build_panel, composition_lines

panel_result = build_panel(spark, survey, storage)
for line in composition_lines(survey, panel_result.selection):
    print(line)

## 3. `screen` — スクリーニングしてパネルを確定する

スクリーナーが無い場合と `assume` では判定するものが無いので何もしない
（`screen_survey` が内部で判定して早期に戻る）。

In [0]:
from persona_sim.panel.screening import screen_survey
from persona_sim.run.progress import console_reporter

# 進捗を渡さないと、判定中は一切の出力が無く、遅いのか止まったのか区別できない。
screening = screen_survey(spark, survey, storage, progress=console_reporter())
print(f"方式: {screening.method}")
print("判定:", screening.sessions_ok, "/", screening.sessions_total, f"(スキップ {screening.sessions_skipped})")
print(f"確定: {screening.achieved} / 予備: {screening.reserve} / 非通過: {screening.screened_out}")
for warning in screening.warnings:
    print("警告", warning)
if screening.aborted_reason:
    # 握りつぶすと、判定できていないだけなのに「条件に合う人がいない」ように見える。
    raise RuntimeError(f"スクリーニングを中断した: {screening.aborted_reason}")

## 4. `run` — 回答を生成する

**中断しても、同じ調査定義でこのノートブックを再実行すれば続きから再開します。**
完了済みのセッションはスキップされます。

In [0]:
from persona_sim.storage import delta
from persona_sim.storage.locator import SCREENER_RESPONSES, locator

(delta.read_table(spark, locator(SCREENER_RESPONSES, storage))
    .filter(f"survey_id = '{survey.survey_id}'")
    .selectExpr("count(*) n", "avg(attempt) avg_attempt",
                "percentile_approx(latency_ms, 0.5) p50",
                "percentile_approx(latency_ms, 0.95) p95", "max(latency_ms) mx")
    .show())


In [0]:
from persona_sim.llm.registry import build_client
from persona_sim.run.progress import console_reporter
from persona_sim.run.run import run_survey

# クライアントを run と進捗表示で共有する。共有しないと進捗行に再試行の状況
# （＝エンドポイントが並列度を捌けていない兆候）を出せない。
# モデルは調査定義から引く（run_survey が内部で行うのと同じ build_client(survey.model)）。
client = build_client(survey.model)

run_result = run_survey(
    spark,
    survey,
    storage,
    client=client,
    progress=console_reporter(client=client),
)
print("セッション:", run_result.sessions_ok, "/", run_result.sessions_total)
print("失敗      :", run_result.sessions_failed)
for warning in run_result.warnings:
    print("警告", warning)
if run_result.sessions_failed:
    raise RuntimeError(
        f"{run_result.sessions_failed} 件のセッションが失敗した: {survey.survey_id}"
    )

## 5. `aggregate` — 集計してファイルを書き出す

`output.formats` に従って CSV / xlsx / Delta を書き出す（`export` の分も含む）。

In [0]:
from persona_sim.aggregate.aggregate import aggregate_survey

aggregate_result = aggregate_survey(spark, survey, storage, output_dir())
for path in aggregate_result.written:
    print(path)

## 6. 実行メタデータを残す

`runs` テーブルと `run_metadata.json` / `prompt_sample.md` を書く（`SPEC.md` §9）。

In [0]:
from persona_sim.metadata import write_metadata

metadata_path = write_metadata(spark, survey, run_result, storage, output_dir())
print("実行メタデータ:", metadata_path)

## 7. トークン使用量

スクリーナーと回答生成で使ったトークンを分けて出します。**この実行で呼び出したぶんだけ**
の数字です（`run_metadata.json` の `tokens_scope: "run"` と同じ範囲）。再開でスキップした
セッションと、判定済みで聞き直さなかった候補は含みません。`assume` やスクリーナー無しの
調査ではスクリーナー側が 0 になります。

金額換算は載せません。単価はワークスペースごとに違い、システム側に持たせると必ず古く
なるためです（`SPEC.md` §9）。トークン数から利用側で換算してください。

In [0]:
import unicodedata


def display_width(text: str) -> int:
    """等幅表示での桁数。全角は2桁。日本語の見出しで表がずれないように。"""
    return sum(2 if unicodedata.east_asian_width(ch) in "WF" else 1 for ch in text)


def ljust(text: str, width: int) -> str:
    return text + " " * max(0, width - display_width(text))


def rjust(text: str, width: int) -> str:
    return " " * max(0, width - display_width(text)) + text


LABEL_WIDTH = 16
VALUE_WIDTH = 14

rows = [
    ("スクリーナー", screening.input_tokens, screening.output_tokens),
    ("回答生成", run_result.input_tokens, run_result.output_tokens),
]
total_input = sum(row[1] for row in rows)
total_output = sum(row[2] for row in rows)
rule = "-" * (LABEL_WIDTH + VALUE_WIDTH * 2)

print(ljust("区分", LABEL_WIDTH) + rjust("入力", VALUE_WIDTH) + rjust("出力", VALUE_WIDTH))
print(rule)
for label, input_tokens, output_tokens in rows:
    print(ljust(label, LABEL_WIDTH) + f"{input_tokens:>{VALUE_WIDTH},}{output_tokens:>{VALUE_WIDTH},}")
print(rule)
print(ljust("合計", LABEL_WIDTH) + f"{total_input:>{VALUE_WIDTH},}{total_output:>{VALUE_WIDTH},}")
print()
print("※ この実行で呼び出したぶんのみ（再開でスキップしたセッション、"
      "判定済みで聞き直さなかった候補は含まない）")